 ### Cell 1 — Install
### This installs Unsloth (a library that makes loading and fine-tuning LLMs faster and
### lighter on memory) along with the other training libraries we need: transformers, peft
### (for LoRA), trl (for the training loop), and bitsandbytes (for 4-bit loading).


In [1]:
!pip install -q unsloth
!pip install -q --upgrade --no-cache-dir "git+https://github.com/unslothai/unsloth.git"
!pip install -q wandb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.3/60.3 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.0/74.0 MB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 31.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.6/915.6 MB 719.0 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 89.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.3/322.3 MB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 127.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 36.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 71.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 118.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3

 ### Cell 2 — Log in to Weights & Biases
### W&B is a free dashboard that records our training progress (loss curves, hyperparameters)
### so we can see whether training is going well, and show the graphs later as portfolio
### evidence.

In [2]:
import wandb
wandb.login()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.


wandb: Paste your API key and hit enter: ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: kaustubh-aggarwal31 (kaustubh-aggarwal31-vellore-institute-of-technology) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

# ### Cell 3 — Load the base model in 4-bit (QLoRA)
### This downloads Llama 3.1 8B and loads it in 4-bit precision, which shrinks its memory
### footprint from ~16GB down to about 5-6GB so it fits on a free T4 GPU. Unsloth handles the
### 4-bit loading automatically and makes training roughly 2x faster than a plain setup.

In [3]:
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LENGTH = 1024   # max tokens per example; our clauses are short so this is plenty
MODEL_NAME = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit"  # pre-quantized 4-bit version

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,          # auto-detects float16 for T4, bfloat16 for newer GPUs (A100/L4)
    load_in_4bit=True,
)

print("Base model loaded in 4-bit.")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.6.9: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.53k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/55.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

Unsloth: Will load unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit as a legacy tokenizer.


Base model loaded in 4-bit.


# ### Cell 4 — Attach LoRA adapters
### The base model's weights are frozen and untouched. Here we attach small trainable
### "adapter" layers on top, settings: r=16 (adapter size), alpha=16
### (how strongly the adapter affects output), applied to all the main linear layers.
### These adapters are the only part of the model that will actually learn anything.

In [4]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    bias="none",
    use_gradient_checkpointing="unsloth",   # saves memory during training
    random_state=42,
)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable_params:,} ({100*trainable_params/total_params:.3f}% of total)")

Unsloth 2026.6.9 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


Trainable params: 41,943,040 (0.915% of total)


# ### Cell 5 — Load our dataset from Step 01
### We load the train.jsonl and val.jsonl files created in Step 01.

In [6]:
from datasets import load_dataset
import json

# Option A: files uploaded directly to this Colab session
train_ds = load_dataset("json", data_files="data/train.jsonl", split="train")
val_ds = load_dataset("json", data_files="data/val.jsonl", split="train")

with open("data/categories.json") as f:
    cat_config = json.load(f)

print(f"Train examples: {len(train_ds)}")
print(f"Val examples:   {len(val_ds)}")
print(f"Categories:     {cat_config['categories']}")

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Train examples: 1200
Val examples:   150
Categories:     ['Governing Laws', 'Terminations', 'Confidentiality', 'Indemnifications', 'Assignments', 'Notices', 'Severability', 'Counterparts', 'Amendments', 'Survival']


# ###Cell 6 — Apply Llama 3.1's chat template
### This is the format-safety: instead of hand-writing Llama's special
### tokens, we use Unsloth's get_chat_template helper to configure the tokenizer with
### Llama 3.1's exact official template, then hand it our system/user/assistant messages.
### This guarantees we never accidentally use the wrong format for this specific model.

In [7]:
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template="llama-3.1",
)

def format_example(example):
    text = tokenizer.apply_chat_template(
        example["messages"], tokenize=False, add_generation_prompt=False
    )
    return {"text": text}

train_ds = train_ds.map(format_example)
val_ds = val_ds.map(format_example)

print("Example formatted training text:\n")
print(train_ds[0]["text"])

Map:   0%|          | 0/1200 [00:00<?, ? examples/s]

Map:   0%|          | 0/150 [00:00<?, ? examples/s]

Example formatted training text:

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 July 2024

You are a legal clause classifier. Read the contract clause and respond with exactly one category from this list: Governing Laws, Terminations, Confidentiality, Indemnifications, Assignments, Notices, Severability, Counterparts, Amendments, Survival. Respond with only the category name and nothing else.<|eot_id|><|start_header_id|>user<|end_header_id|>

Clause:
Notwithstanding anything to the contrary in this Agreement, the obligations of Sellers set forth in this Article VII shall survive until sixty days following the expiration of the statute of limitations for the Tax liability in question.<|eot_id|><|start_header_id|>assistant<|end_header_id|>

Survival<|eot_id|>


# ### Cell 7 — Configure and run SFT training
### This sets up the actual training loop: how many examples per batch, how many times to
### go through the full dataset (epochs), and how big a correction to make each step
### (learning rate). SFTTrainer (from the trl library) handles the training loop itself —
### we just give it our settings and formatted data.

In [8]:
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir="outputs_sft",
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,   # effective batch size = 8*2 = 16
    num_train_epochs=3,
    learning_rate=2e-4,
    warmup_steps=10,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=25,
    save_strategy="epoch",
    optim="adamw_8bit",              # memory-efficient optimizer
    seed=42,
    report_to="wandb",
    run_name="legal-clause-sft-llama3.1-8b",
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    packing=False,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    args=training_args,
)

trainer_stats = trainer.train()

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/1200 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/150 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,200 | Num Epochs = 3 | Total steps = 225
O^O/ \_/ \    Batch size per device = 8 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (8 x 2 x 1) = 16
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)


wandb: Detected [huggingface_hub.inference, openai] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss,Validation Loss
25,0.966796,0.767935
50,0.705165,0.651121
75,0.666507,0.623614
100,0.608030,0.622202
125,0.622469,0.609780
150,0.560777,0.603246
175,0.531815,0.620498
200,0.513600,0.620493
225,0.487653,0.616470


Unsloth: Restored added_tokens_decoder metadata in outputs_sft/checkpoint-75/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs_sft/checkpoint-150/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs_sft/checkpoint-225/tokenizer_config.json.


# ### Cell 8 — Save the trained LoRA adapter
### Training only updated the small adapter layers, so we only need to save those (typically
### under 200MB), not the whole 8B model. This adapter is the actual deliverable of this step
### — Step 03 (DPO) will load the base model + this adapter and continue training it further.

In [9]:
ADAPTER_DIR = "llama3.1-8b-legal-clause-lora-sft"
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

print(f"Adapter saved to: {ADAPTER_DIR}")
print("Step 02 complete. Ready for Step 03 (DPO alignment).")

Unsloth: Restored added_tokens_decoder metadata in llama3.1-8b-legal-clause-lora-sft/tokenizer_config.json.


Adapter saved to: llama3.1-8b-legal-clause-lora-sft
Step 02 complete. Ready for Step 03 (DPO alignment).


In [10]:
!zip -r llama3.1-8b-legal-clause-lora-sft.zip llama3.1-8b-legal-clause-lora-sft

  adding: llama3.1-8b-legal-clause-lora-sft/ (stored 0%)
  adding: llama3.1-8b-legal-clause-lora-sft/chat_template.jinja (deflated 72%)
  adding: llama3.1-8b-legal-clause-lora-sft/adapter_config.json (deflated 58%)
  adding: llama3.1-8b-legal-clause-lora-sft/adapter_model.safetensors (deflated 7%)
  adding: llama3.1-8b-legal-clause-lora-sft/tokenizer_config.json (deflated 96%)
  adding: llama3.1-8b-legal-clause-lora-sft/README.md (deflated 65%)
  adding: llama3.1-8b-legal-clause-lora-sft/tokenizer.json (deflated 85%)


In [11]:
from google.colab import drive
drive.mount('/content/drive')
!cp -r llama3.1-8b-legal-clause-lora-sft /content/drive/MyDrive/

Mounted at /content/drive
